### Import Necessary dependencies

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id # this creates a column for indexing/number increment
import pandas as pd
from sqlalchemy.engine import URL
from sqlalchemy import create_engine
import psycopg2
print(psycopg2.__version__)


2.9.11 (dt dec pq3 ext lo64)


In [4]:
# Initialise the spark session
spark = SparkSession.builder.appName('NugaBankETL').getOrCreate()

In [5]:
spark

### Data Extraction

In [6]:
nuga_bank_df = spark.read.csv('dataset/nuga_bank.csv', header=True, inferSchema=True)

In [7]:
nuga_bank_df.show(5)

+--------------------+------+----------------+--------------+--------------------+------------------+--------------+--------------------+--------------------+--------------------+--------------------+-------------------+------------------+--------------------+-------------+-------------+--------+-----+---------+--------------------+--------------------+------+--------------+
|    Transaction_Date|Amount|Transaction_Type| Customer_Name|    Customer_Address|     Customer_City|Customer_State|    Customer_Country|             Company|           Job_Title|               Email|       Phone_Number|Credit_Card_Number|                IBAN|Currency_Code|Random_Number|Category|Group|Is_Active|        Last_Updated|         Description|Gender|Marital_Status|
+--------------------+------+----------------+--------------+--------------------+------------------+--------------+--------------------+--------------------+--------------------+--------------------+-------------------+------------------+-----

In [8]:
# view schema in pyspark
nuga_bank_df.printSchema()

root
 |-- Transaction_Date: timestamp (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Transaction_Type: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Customer_Address: string (nullable = true)
 |-- Customer_City: string (nullable = true)
 |-- Customer_State: string (nullable = true)
 |-- Customer_Country: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone_Number: string (nullable = true)
 |-- Credit_Card_Number: long (nullable = true)
 |-- IBAN: string (nullable = true)
 |-- Currency_Code: string (nullable = true)
 |-- Random_Number: double (nullable = true)
 |-- Category: string (nullable = true)
 |-- Group: string (nullable = true)
 |-- Is_Active: string (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)
 |-- Description: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Marital_Status: string (nullable = true)

In [9]:
# Data Cleaning and Transformation
# View colums
nuga_bank_df.columns

['Transaction_Date',
 'Amount',
 'Transaction_Type',
 'Customer_Name',
 'Customer_Address',
 'Customer_City',
 'Customer_State',
 'Customer_Country',
 'Company',
 'Job_Title',
 'Email',
 'Phone_Number',
 'Credit_Card_Number',
 'IBAN',
 'Currency_Code',
 'Random_Number',
 'Category',
 'Group',
 'Is_Active',
 'Last_Updated',
 'Description',
 'Gender',
 'Marital_Status']

In [10]:
# check number of rows
num_rows = nuga_bank_df.count()
print(num_rows)

1000000


In [11]:
# view number of columns
num_columns = len(nuga_bank_df.columns)

num_columns

23

In [12]:
# checking for null values
for column in nuga_bank_df.columns:
    print(column, 'Nulls', nuga_bank_df.filter(nuga_bank_df[column].isNull()).count())

Transaction_Date Nulls 0
Amount Nulls 0
Transaction_Type Nulls 0
Customer_Name Nulls 100425
Customer_Address Nulls 100087
Customer_City Nulls 100034
Customer_State Nulls 100009
Customer_Country Nulls 100672
Company Nulls 100295
Job_Title Nulls 99924
Email Nulls 100043
Phone_Number Nulls 100524
Credit_Card_Number Nulls 100085
IBAN Nulls 100300
Currency_Code Nulls 99342
Random_Number Nulls 99913
Category Nulls 100332
Group Nulls 100209
Is_Active Nulls 100259
Last_Updated Nulls 100321
Description Nulls 100403
Gender Nulls 99767
Marital_Status Nulls 99904


In [13]:
# How to Fil up missing values
nuga_bank_df_clean = nuga_bank_df.fillna({
    'Customer_Name' : 'Unknown',
    'Customer_Address' : 'Unknown',
    'Customer_City' : 'Unknown',
    'Customer_State': 'Unknown',
    'Customer_Country' : 'Unknown',
    'Company' : 'Unknown',
    'Job_Title' : 'Unknown',
    'Email' : 'Unknown',
    'Phone_Number' : 'Unknown',
    'Credit_Card_Number' : 0,
    'IBAN' : 'Unknown',
    'Currency_Code' : 'Unknown',
    'Random_Number' : 0.0,
    'Category' : 'Unknown',
    'Group' : 'Unknown',
    'Is_Active' : 'Unknown',
    'Description': 'Unknown',
    'Gender' : 'Unknown',
    'Marital_Status' : 'Unknown'

})

In [14]:
# checking for null values
for column in nuga_bank_df_clean.columns:
    print(column, 'Nulls', nuga_bank_df_clean.filter(nuga_bank_df_clean[column].isNull()).count())

Transaction_Date Nulls 0
Amount Nulls 0
Transaction_Type Nulls 0
Customer_Name Nulls 0
Customer_Address Nulls 0
Customer_City Nulls 0
Customer_State Nulls 0
Customer_Country Nulls 0
Company Nulls 0
Job_Title Nulls 0
Email Nulls 0
Phone_Number Nulls 0
Credit_Card_Number Nulls 0
IBAN Nulls 0
Currency_Code Nulls 0
Random_Number Nulls 0
Category Nulls 0
Group Nulls 0
Is_Active Nulls 0
Last_Updated Nulls 100321
Description Nulls 0
Gender Nulls 0
Marital_Status Nulls 0


In [15]:
# Drop rows where last_updated is null
nuga_bank_df_clean = nuga_bank_df_clean.na.drop(subset= ['Last_Updated'])

In [16]:
# checking for null values
for column in nuga_bank_df_clean.columns:
    print(column, 'Nulls', nuga_bank_df_clean.filter(nuga_bank_df_clean[column].isNull()).count())

Transaction_Date Nulls 0
Amount Nulls 0
Transaction_Type Nulls 0
Customer_Name Nulls 0
Customer_Address Nulls 0
Customer_City Nulls 0
Customer_State Nulls 0
Customer_Country Nulls 0
Company Nulls 0
Job_Title Nulls 0
Email Nulls 0
Phone_Number Nulls 0
Credit_Card_Number Nulls 0
IBAN Nulls 0
Currency_Code Nulls 0
Random_Number Nulls 0
Category Nulls 0
Group Nulls 0
Is_Active Nulls 0
Last_Updated Nulls 0
Description Nulls 0
Gender Nulls 0
Marital_Status Nulls 0


In [17]:
# Check total number of rows after dropping
num_row_clean = nuga_bank_df_clean.count()

num_row_clean

899679

In [18]:
# To view the summary statistics of the data
nuga_bank_df_clean.describe().show()

+-------+------------------+----------------+-------------+--------------------+-------------+--------------+----------------+-------------+------------------+-------------------+--------------------+--------------------+--------------------+-------------+------------------+--------+-------+---------+--------------------+-------+--------------+
|summary|            Amount|Transaction_Type|Customer_Name|    Customer_Address|Customer_City|Customer_State|Customer_Country|      Company|         Job_Title|              Email|        Phone_Number|  Credit_Card_Number|                IBAN|Currency_Code|     Random_Number|Category|  Group|Is_Active|         Description| Gender|Marital_Status|
+-------+------------------+----------------+-------------+--------------------+-------------+--------------+----------------+-------------+------------------+-------------------+--------------------+--------------------+--------------------+-------------+------------------+--------+-------+---------+----

In [19]:
nuga_bank_df_clean.columns

['Transaction_Date',
 'Amount',
 'Transaction_Type',
 'Customer_Name',
 'Customer_Address',
 'Customer_City',
 'Customer_State',
 'Customer_Country',
 'Company',
 'Job_Title',
 'Email',
 'Phone_Number',
 'Credit_Card_Number',
 'IBAN',
 'Currency_Code',
 'Random_Number',
 'Category',
 'Group',
 'Is_Active',
 'Last_Updated',
 'Description',
 'Gender',
 'Marital_Status']

### Let's carry out Transformation as we have created data model

In [40]:
# transaction table
transaction = nuga_bank_df_clean.select('Transaction_Date', 'Amount', 'Transaction_Type').distinct()

In [41]:
# Adding the transaction_id column
transaction = transaction.withColumn('transaction_id', monotonically_increasing_id())

In [43]:
# reordering the column
transaction = transaction.select('transaction_id', 'Transaction_Date', 'Amount', 'Transaction_Type' )

In [44]:
transaction.show()

+--------------+--------------------+------+----------------+
|transaction_id|    Transaction_Date|Amount|Transaction_Type|
+--------------+--------------------+------+----------------+
|             0|2024-02-06 09:21:...|394.54|         Deposit|
|             1|2024-02-06 22:04:...|212.45|         Deposit|
|             2|2024-02-18 21:04:...|521.62|         Deposit|
|             3|2024-04-11 13:35:...|416.11|         Deposit|
|             4|2024-01-09 17:20:...|293.14|         Deposit|
|             5|2024-01-17 10:42:...|790.44|         Deposit|
|             6|2024-03-21 04:05:...|329.22|         Deposit|
|             7|2024-04-29 10:42:...| 28.27|        Transfer|
|             8|2024-04-27 01:11:...| 32.36|      Withdrawal|
|             9|2024-02-16 15:28:...|770.79|      Withdrawal|
|            10|2024-01-23 23:58:...|531.71|      Withdrawal|
|            11|2024-04-13 04:39:...| 152.8|         Deposit|
|            12|2024-02-25 13:29:...|294.54|      Withdrawal|
|       

In [ ]:
# customer table
customer = nuga_bank_df_clean.select('Customer_Name',
 'Customer_Address',
 'Customer_City',
 'Customer_State',
 'Customer_Country',
 'Email',
 'Phone_Number').distinct()  #distinct() help to reduce redundancy/repetition of rows

# Add customer_id column
customer = customer.withColumn('customer_id', monotonically_increasing_id())

# Reorder the table
customer = customer.select('customer_id', 'Customer_Name',
 'Customer_Address',
 'Customer_City',
 'Customer_State',
 'Customer_Country',
 'Email',
 'Phone_Number')

customer.show()

+-----------+------------------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+
|customer_id|     Customer_Name|    Customer_Address|       Customer_City|Customer_State|    Customer_Country|               Email|        Phone_Number|
+-----------+------------------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+
|          0|    Miguel Leonard|262 Beck Expressw...|             Unknown| West Virginia|             Eritrea| zweaver@example.net|             Unknown|
|          1|           Unknown|             Unknown|         Evanchester|        Oregon|             Uruguay|             Unknown| (384)778-9942x91236|
|          2|    Michael Murphy|894 Williams Ridg...|       Dominguezview|      New York|              Sweden|kristinstanley@ex...|+1-693-739-2204x8851|
|          3|    Tina Gutierrez|    415 Taylor Knoll|           Donnastad|South Ca

In [ ]:
# employee table
employee = nuga_bank_df_clean.select('Company',
 'Job_Title',
 'Gender',
 'Marital_Status').distinct()

# Add id_colum
employee = employee.withColumn('employee_id', monotonically_increasing_id())

# re-order the dataframe
employee = employee.select('employee_id', 
                           'Company',
                            'Job_Title',
                            'Gender',
                            'Marital_Status')

In [39]:
employee.count()

809052

In [47]:
# fact table
fact_table = nuga_bank_df_clean.join(customer, ['Customer_Name', 'Customer_Address', 'Customer_City', 'Customer_State', 
                                                'Customer_Country', 'Email', 'Phone_Number'], how='left') \
                                .join(transaction, ['Transaction_Date', 'Amount', 'Transaction_Type'], how='left') \
                                .join(employee, ['Company', 'Job_Title', 'Gender', 'Marital_Status'], how='left') \
                                .select('transaction_id', 'customer_id', 'employee_id', 'Credit_Card_Number',
                                        'IBAN',
                                        'Currency_Code',
                                        'Random_Number',
                                        'Category',
                                        'Group',
                                        'Is_Active',
                                        'Last_Updated',
                                        'Description')
 

In [48]:
fact_table.show()

+--------------+-----------+-----------+-------------------+--------------------+-------------+-------------+--------+-------+---------+--------------------+--------------------+
|transaction_id|customer_id|employee_id| Credit_Card_Number|                IBAN|Currency_Code|Random_Number|Category|  Group|Is_Active|        Last_Updated|         Description|
+--------------+-----------+-----------+-------------------+--------------------+-------------+-------------+--------+-------+---------+--------------------+--------------------+
|   34359755330|68719480203|      16979|4191948926153525351|GB33SCRV040476482...|          MRO|       4711.0|       C|Unknown|       No|2023-03-30 22:09:...|Program dinner me...|
|         61026|42949729294|      58467|    180029146062238|GB17MTZS784426277...|          GIP|       5688.0|       A|Unknown|  Unknown|2021-02-23 05:57:...|Network quite sit...|
|   34359821010|42949747042|      66666|   4596307291033331|GB59TAVO484072449...|          ERN|       713

### Outputing the transformed data to parquet

In [52]:
transaction.write.mode('overwrite').parquet('dataset/transaction')
transaction.write.mode('overwrite').parquet('dataset/customer')
transaction.write.mode('overwrite').parquet('dataset/employee')
transaction.write.mode('overwrite').parquet('dataset/fact_table')

Py4JJavaError: An error occurred while calling o604.parquet.
: java.util.concurrent.ExecutionException: Boxed Exception
	at scala.concurrent.impl.Promise$.scala$concurrent$impl$Promise$$resolve(Promise.scala:99)
	at scala.concurrent.impl.Promise$DefaultPromise.tryComplete(Promise.scala:278)
	at scala.concurrent.Promise.complete(Promise.scala:57)
	at scala.concurrent.Promise.complete$(Promise.scala:56)
	at scala.concurrent.impl.Promise$DefaultPromise.complete(Promise.scala:104)
	at scala.concurrent.Promise.failure(Promise.scala:109)
	at scala.concurrent.Promise.failure$(Promise.scala:109)
	at scala.concurrent.impl.Promise$DefaultPromise.failure(Promise.scala:104)
	at org.apache.spark.sql.execution.adaptive.ResultQueryStageExec.$anonfun$doMaterialize$2(QueryStageExec.scala:333)
	at java.base/java.util.concurrent.CompletableFuture.uniWhenComplete(CompletableFuture.java:863)
	at java.base/java.util.concurrent.CompletableFuture$UniWhenComplete.tryFire(CompletableFuture.java:841)

In [65]:
# Out put the transformed data as csv
transaction.repartition(3).write.mode('overwrite').option('header', 'true').csv('dataset/tranformeddata/csv/transaction')
transaction.repartition(3).write.mode('overwrite').option('header', 'true').csv('dataset/tranformeddata/csv/customer')
transaction.repartition(3).write.mode('overwrite').option('header', 'true').csv('dataset/tranformeddata/csv/employee')
transaction.repartition(3).write.mode('overwrite').option('header', 'true').csv('dataset/tranformeddata/csv/fact_table')


Py4JJavaError: An error occurred while calling o636.csv.
: java.util.concurrent.ExecutionException: Boxed Exception
	at scala.concurrent.impl.Promise$.scala$concurrent$impl$Promise$$resolve(Promise.scala:99)
	at scala.concurrent.impl.Promise$DefaultPromise.tryComplete(Promise.scala:278)
	at scala.concurrent.Promise.complete(Promise.scala:57)
	at scala.concurrent.Promise.complete$(Promise.scala:56)
	at scala.concurrent.impl.Promise$DefaultPromise.complete(Promise.scala:104)
	at scala.concurrent.Promise.failure(Promise.scala:109)
	at scala.concurrent.Promise.failure$(Promise.scala:109)
	at scala.concurrent.impl.Promise$DefaultPromise.failure(Promise.scala:104)
	at org.apache.spark.sql.execution.adaptive.ResultQueryStageExec.$anonfun$doMaterialize$2(QueryStageExec.scala:333)
	at java.base/java.util.concurrent.CompletableFuture.uniWhenComplete(CompletableFuture.java:863)
	at java.base/java.util.concurrent.CompletableFuture$UniWhenComplete.tryFire(CompletableFuture.java:841)
	at

### Convert spark dataFrame to Pandas dataFrame


In [53]:
transaction_pd_df = transaction.toPandas()
customer_pd_df = customer.toPandas()
employee_pd_df = employee.toPandas()
fact_table_pd_df = fact_table.toPandas()

### Loading the database into a PostgreSql DB

In [62]:
uid = 'postgres'
pwd = 'password'
server = 'localhost'
database = 'nuga_bank'

engine = create_engine(f'postgresql://{uid}:{pwd}@{server}:5432/{database}')

#Connect to PostgreSQL server
with engine.connect() as connection:
    # Create tables and load the data
    transaction_pd_df.to_sql('transaction', connection, index=False, if_exists='replace')
    customer_pd_df.to_sql('customer', connection, index=False, if_exists='replace')
    employee_pd_df.to_sql('employee', connection, index=False, if_exists='replace')
    fact_table_pd_df.to_sql('fact_table', connection, index=False, if_exists='replace')

print('Database, tables and data loaded successfully! ' )

Database, tables and data loaded successfully! 
